# Credit Card Fraud Detection
## End-to-End Machine Learning Pipeline

**Problem Type:** Binary Classification  
**Dataset:** Kaggle Credit Card Fraud Detection (mlg-ulb/creditcardfraud)  
**Key Challenge:** Severe class imbalance — only 0.17% of transactions are fraudulent

---
## Phase 1 — Problem Definition

**Objective:** Build a binary classifier that identifies whether a credit card transaction is fraudulent (Class=1) or legitimate (Class=0).

**Input Features:**
- `Time` — Seconds elapsed since the first transaction
- `V1` through `V28` — Anonymized PCA-transformed features
- `Amount` — Transaction amount in USD

**Target Variable:** `Class` — 0 = Legitimate, 1 = Fraud

**Why accuracy is misleading:** A model predicting every transaction as legitimate achieves 99.83% accuracy yet catches zero fraud. **Recall** and **F1-Score** are used as the primary evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('All libraries imported successfully.')

---
## Phase 2 — Dataset Loading and Understanding

In [ ]:
# Mount Google Drive and load dataset
# Upload creditcard.csv to your Google Drive root (My Drive) before running this cell.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    csv_path = '/content/drive/MyDrive/creditcard.csv'
    print('Google Drive mounted. Loading from Drive...')
except Exception:
    csv_path = 'creditcard.csv'
    print('Running locally. Loading from current directory...')

df = pd.read_csv(csv_path)
print('Dataset loaded successfully.')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print()
df.info()

In [ ]:
class_counts = df['Class'].value_counts()
class_pct    = df['Class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  Legitimate (0): {class_counts[0]:,}  ({class_pct[0]:.2f}%)')
print(f'  Fraud      (1): {class_counts[1]:,}  ({class_pct[1]:.4f}%)')
print(f'  Total:          {len(df):,}')
print(f'  Imbalance ratio: {class_counts[0]//class_counts[1]}:1')

---
## Phase 3 — Data Preprocessing

In [ ]:
# Step 1: Check missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')
print(f'Total missing: {missing.sum()}')

# Step 2: Check duplicates
dup = df.duplicated().sum()
print(f'\nDuplicate rows: {dup}')
if dup > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates to remove.')

In [ ]:
# Step 3: Outlier detection on Amount using IQR method
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['Amount'] < lower) | (df['Amount'] > upper)]
print(f'IQR range for Amount: [{lower:.2f}, {upper:.2f}]')
print(f'Outliers detected: {len(outliers):,} rows ({len(outliers)/len(df)*100:.2f}%)')
print('Outliers are retained — high-value legitimate transactions exist in real data.')

In [ ]:
# Step 4: Normalize Amount and Time using StandardScaler
# V1-V28 are already normalized via PCA — no scaling needed for them.
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

# Save scaler fitted on [Time, Amount] for use in app.py
amount_time_scaler = StandardScaler()
amount_time_scaler.fit(df[['Time', 'Amount']])
joblib.dump(amount_time_scaler, 'scaler.pkl')
print('scaler.pkl saved successfully.')

# Step 5: Build feature matrix
features = ['Time_scaled'] + [f'V{i}' for i in range(1, 29)] + ['Amount_scaled']
X = df[features]
y = df['Class']
print('Feature matrix shape:', X.shape)
print('Features (in order):', X.columns.tolist())

# Step 6: Train-test split (stratified to preserve fraud ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape} | Fraud: {y_train.sum()}')
print(f'Test:  {X_test.shape}  | Fraud: {y_test.sum()}')

# Step 7: SMOTE — applied only on training data to prevent data leakage
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print('\nBefore SMOTE:', Counter(y_train))
print('After SMOTE: ', Counter(y_train_res))

---
## Phase 4 — Exploratory Data Analysis

In [ ]:
print('Statistical Summary:')
df[['Time', 'Amount', 'Class']].describe().round(2)

In [ ]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

# Plot 1: Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(legit['Amount'], bins=80, color='#2196F3', alpha=0.6, label='Legitimate', density=True)
axes[0].hist(fraud['Amount'], bins=80, color='#F44336', alpha=0.8, label='Fraud', density=True)
axes[0].set_xlabel('Transaction Amount (USD)')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Amount Distribution (Fraud vs Legitimate)')
axes[0].legend()
axes[0].set_xlim(0, 2000)

axes[1].hist(fraud['Amount'], bins=50, color='#F44336', edgecolor='#B71C1C', linewidth=0.5)
axes[1].set_xlabel('Transaction Amount (USD)')
axes[1].set_ylabel('Count')
axes[1].set_title('Fraud Transaction Amount Distribution (Zoomed)')

plt.tight_layout()
plt.savefig('plot_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fraud — mean: ${fraud["Amount"].mean():.2f}, median: ${fraud["Amount"].median():.2f}')
print(f'Legit — mean: ${legit["Amount"].mean():.2f}, median: ${legit["Amount"].median():.2f}')

In [ ]:
# Plot 2: Boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bp = axes[0].boxplot([legit['Amount'], fraud['Amount']],
                     labels=['Legitimate', 'Fraud'],
                     patch_artist=True,
                     flierprops=dict(marker='o', markersize=2, alpha=0.3))
bp['boxes'][0].set_facecolor('#BBDEFB')
bp['boxes'][1].set_facecolor('#FFCDD2')
axes[0].set_ylabel('Transaction Amount (USD)')
axes[0].set_title('Amount Distribution by Class (with outliers)')

bp2 = axes[1].boxplot([np.log1p(legit['Amount']), np.log1p(fraud['Amount'])],
                      labels=['Legitimate', 'Fraud'], patch_artist=True,
                      flierprops=dict(marker='o', markersize=2, alpha=0.4))
bp2['boxes'][0].set_facecolor('#BBDEFB')
bp2['boxes'][1].set_facecolor('#FFCDD2')
axes[1].set_ylabel('log(Amount + 1)')
axes[1].set_title('Log-Scaled Amount by Class')

plt.tight_layout()
plt.savefig('plot_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Heatmap
corr_cols = ['Time', 'Amount', 'V1', 'V2', 'V3', 'V4', 'V5',
             'V7', 'V10', 'V11', 'V12', 'V14', 'V17', 'V19', 'Class']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix (selected features + target)', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('plot_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

class_corr = df[[f'V{i}' for i in range(1,29)] + ['Amount','Time']].corrwith(df['Class'])
print('Top 10 features correlated with fraud:')
print(class_corr.abs().sort_values(ascending=False).head(10))

In [ ]:
# Plot 4: Scatter
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

legit_sample = legit.sample(5000, random_state=42)
axes[0].scatter(legit_sample['Time'], legit_sample['Amount'],
                c='#2196F3', alpha=0.3, s=5, label='Legitimate')
axes[0].scatter(fraud['Time'], fraud['Amount'],
                c='#F44336', alpha=0.7, s=15, label='Fraud', zorder=5)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amount (USD)')
axes[0].set_title('Time vs Amount by Class')
axes[0].legend(markerscale=2)

axes[1].scatter(fraud['Time'], np.log1p(fraud['Amount']),
                c='#F44336', alpha=0.6, s=20,
                edgecolors='#B71C1C', linewidth=0.3)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('log(Amount + 1)')
axes[1].set_title('Fraud Transactions — Time vs Log Amount')

plt.tight_layout()
plt.savefig('plot_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(['Legitimate', 'Fraud'],
                   [class_counts[0], class_counts[1]],
                   color=['#2196F3', '#F44336'], width=0.5, edgecolor='white')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Class Distribution (Absolute Count)')
for bar, count in zip(bars, [class_counts[0], class_counts[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 1000,
                f'{count:,}', ha='center', fontsize=10)

axes[1].pie([class_counts[0], class_counts[1]],
            labels=['Legitimate\n99.83%', 'Fraud\n0.17%'],
            colors=['#2196F3', '#F44336'],
            explode=[0, 0.1], autopct='%1.2f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Class Distribution (Percentage)')

plt.tight_layout()
plt.savefig('plot_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 5 — Feature Engineering

In [ ]:
# Log transformation of Amount reduces right-skew
df['log_amount'] = np.log1p(df['Amount'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['Amount'], bins=100, color='#FF9800', edgecolor='white', linewidth=0.3)
axes[0].set_title('Original Amount (right-skewed)')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Count')
axes[0].set_xlim(0, 2000)

axes[1].hist(df['log_amount'], bins=100, color='#4CAF50', edgecolor='white', linewidth=0.3)
axes[1].set_title('log(Amount + 1) — more symmetric')
axes[1].set_xlabel('log(Amount + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plot_log_amount.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Amount skewness:     {df["Amount"].skew():.3f}')
print(f'log_amount skewness: {df["log_amount"].skew():.3f}  (closer to 0 is better)')

In [ ]:
# Feature importance using quick Random Forest
quick_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, max_depth=8)
quick_rf.fit(X_train_res, y_train_res)

importances = pd.Series(quick_rf.feature_importances_, index=X_train_res.columns)
importances_sorted = importances.sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#F44336' if imp > importances_sorted.quantile(0.75) else '#90CAF9'
          for imp in importances_sorted.values]
ax.barh(importances_sorted.index, importances_sorted.values, color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance Score')
ax.set_title('Top 20 Feature Importances (red = top quartile)')
ax.axvline(x=importances_sorted.quantile(0.75), color='gray',
           linestyle='--', alpha=0.5, label='75th percentile')
ax.legend()
plt.tight_layout()
plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most important features:')
print(importances.sort_values(ascending=False).head(10).to_string())

---
## Phase 6 — Model Building

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = {
        'Model':     model_name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4)
    }
    return metrics, y_pred, y_prob

print('Evaluation helper defined.')

In [ ]:
# Model 1: Logistic Regression
# Justification: Simple interpretable baseline for binary classification.
# class_weight='balanced' compensates for the severe imbalance in the original data.
print('Training Logistic Regression...')
lr = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=42, solver='lbfgs', C=0.01
)
lr.fit(X_train_res, y_train_res)
lr_metrics, lr_pred, lr_prob = evaluate_model(lr, X_test, y_test, 'Logistic Regression')
print('Logistic Regression metrics:')
for k, v in lr_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Model 2: Random Forest
# Justification: Ensemble of trees robust to overfitting and class imbalance.
# Provides reliable feature importances for model interpretability.
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf.fit(X_train_res, y_train_res)
rf_metrics, rf_pred, rf_prob = evaluate_model(rf, X_test, y_test, 'Random Forest')
print('Random Forest metrics:')
for k, v in rf_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Model 3: XGBoost
# Justification: Gradient boosting that builds trees sequentially, each correcting
# previous errors. scale_pos_weight handles imbalance directly in the loss function.
# State-of-the-art performance on tabular fraud detection tasks.
print('Training XGBoost...')
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
xgb = XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pos_weight,
    max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss',
    use_label_encoder=False
)
xgb.fit(X_train_res, y_train_res)
xgb_metrics, xgb_pred, xgb_prob = evaluate_model(xgb, X_test, y_test, 'XGBoost')
print('XGBoost metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v}')

---
## Phase 7 — Model Evaluation and Comparison

In [ ]:
results_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics]).set_index('Model')
print('Model Performance Comparison:')
print('=' * 65)
print(results_df.to_string())
print('=' * 65)
print('Best by F1-Score:', results_df['F1-Score'].idxmax())
print('Best by Recall:  ', results_df['Recall'].idxmax())
print('Best by ROC-AUC: ', results_df['ROC-AUC'].idxmax())
print()
print('NOTE: Accuracy is misleading on this dataset (99.83% legitimate).')
print('A trivial all-negative model achieves 99.83% accuracy but catches zero fraud.')
print('Recall and F1-Score are the meaningful evaluation metrics.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

models_data = [
    ('Logistic Regression', lr_pred),
    ('Random Forest',       rf_pred),
    ('XGBoost',             xgb_pred),
]

for ax, (name, pred) in zip(axes, models_data):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'],
                linewidths=1, linecolor='white')
    ax.set_title(f'{name}\nConfusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('plot_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_probs = [
    ('Logistic Regression', lr_prob,  '#1565C0'),
    ('Random Forest',       rf_prob,  '#2E7D32'),
    ('XGBoost',             xgb_prob, '#E65100'),
]

for name, prob, color in model_probs:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(loc='lower right', fontsize=9)

bars = axes[1].bar(results_df.index, results_df['F1-Score'],
                   color=['#1565C0', '#2E7D32', '#E65100'],
                   width=0.5, edgecolor='white')
for bar, val in zip(bars, results_df['F1-Score']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=10)
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1-Score Comparison')
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('plot_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_name = results_df['F1-Score'].idxmax()
best_pred_map = {'Logistic Regression': lr_pred, 'Random Forest': rf_pred, 'XGBoost': xgb_pred}
best_obj_map  = {'Logistic Regression': lr,      'Random Forest': rf,      'XGBoost': xgb}

print(f'Classification Report — {best_name} (Best Model):')
print('=' * 55)
print(classification_report(y_test, best_pred_map[best_name],
                             target_names=['Legitimate', 'Fraud']))

joblib.dump(best_obj_map[best_name], 'fraud_model.pkl')
print(f'fraud_model.pkl saved  ({best_name})')
print('scaler.pkl already saved.')
print('Both files are required to run app.py.')

---
## Summary

| Phase | Action | Key Decision |
|---|---|---|
| Problem Definition | Binary classification | Fraud=1, Legitimate=0 |
| Data Loading | 284,807 rows, 31 columns | Verified structure and class counts |
| Preprocessing | No missing values, no duplicates, IQR check | Outliers retained |
| Normalization | StandardScaler on Time and Amount | V1–V28 already normalized via PCA |
| Train-Test Split | 80/20, stratified | Preserves 0.17% fraud ratio |
| SMOTE | Training data only | Prevents data leakage |
| EDA | Histogram, Boxplot, Heatmap, Scatter | Key features: V4, V11, V14, V17 |
| Feature Engineering | log_amount, feature importance | Reduced Amount skewness |
| Model Training | LR, Random Forest, XGBoost | All trained on SMOTE-resampled data |
| Evaluation | Accuracy, Precision, Recall, F1, AUC | F1 and Recall are primary metrics |
| Deployment | fraud_model.pkl + scaler.pkl | Loaded in Streamlit app |